In [4]:
import torch
from transformers import CLIPProcessor, CLIPModel
from transformers import AutoProcessor, AutoModel
from PIL import Image
import matplotlib.pyplot as plt
import shapiq 
import src
import open_clip

import src
import src.game_huggingface
import src.game_openclip

import os

In [ ]:
from datasets import load_dataset
from huggingface_hub import login

login(token=os.getenv("HF_TOKEN"))

ds = load_dataset("redlessone/Derm1M")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [10]:
ds["train"][0]

{'filename': 'IIYI/26549_2.png',
 'caption': 'Male, 15 years old, has been experiencing spontaneous blistering all over his body since childhood, which heals and forms flat scars as shown in the attached image. The self-assumed diagnosis is dystrophic epidermolysis bullosa, based on a 10-year-old photo with no other available information. Other users and specialists in the forum agree with the diagnosis of dystrophic epidermolysis bullosa, although only photographic evidence was used to support this conclusion.',
 'truncated_caption': 'Male, 15 years old, has been experiencing spontaneous blistering all over his body since childhood, which heals and forms flat scars as shown in the attached image. The self-assumed diagnosis is dystrophic epidermolysis bullosa, based on a 10-year-old photo with no other available information. Other users and specialists in the forum agree with the diagnosis of dystrophic epidermolysis bullosa, although only photographic evidence was used to support this

In [11]:
# LOAD DERMLIP
model_dermlip, _, preprocess_dermlip = open_clip.create_model_and_transforms('hf-hub:redlessone/DermLIP_ViT-B-16')
model_dermlip.eval()

image = preprocess_dermlip(Image.open("tested_images/0d_59_PMC4458964_IJD_60_321e_g003_0.png")).unsqueeze(0)
# Initialize tokenizer
tokenizer_dermlip = open_clip.get_tokenizer('hf-hub:redlessone/DermLIP_ViT-B-16')

# Read example image

# Define disease labels (example: PAD dataset classes)
PAD_CLASSNAMES = [
    "nevus",
    "basal cell carcinoma",
    "actinic keratosis",
    "seborrheic keratosis",
    "squamous cell carcinoma",
    "melanoma"
]

# Build text prompts
template = lambda c: f'This is a skin image of {c}'
text = tokenizer_dermlip([template(c) for c in PAD_CLASSNAMES])

# Inference
with torch.no_grad(), torch.autocast("cuda"):
    # Encode image and text
    image_features = model_dermlip.encode_image(image)
    text_features = model_dermlip.encode_text(text)
    
    # Normalize features
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)
    
    # Compute similarity
    text_probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)

# Get prediction
final_prediction = PAD_CLASSNAMES[torch.argmax(text_probs[0])]
print(f'This image is diagnosed as {final_prediction}.')
print("Label probabilities:", text_probs)

This image is diagnosed as nevus.
Label probabilities: tensor([[0.4391, 0.0087, 0.0382, 0.2123, 0.0077, 0.2939]])


In [12]:
from huggingface_hub import hf_hub_download
import zipfile
import io

def fetch_image_from_hf_zip(filename, repo_id="redlessone/Derm1M", token=True):
    # Derm1M structures images into different zips based on origin.
    # We can infer the correct ZIP based on the filename, or you can hardcode one.
    # E.g., PMC often maps to 'pubmed.zip'
    if "PMC" in filename:
        zip_name = "pubmed.zip"
    elif "tw_" in filename or "twitter" in filename:
        zip_name = "twitter.zip"
    elif "re_" in filename or "reddit" in filename:
        zip_name = "reddit.zip"
    else:
        # Fallback to a common zip if the structure isn't obvious
        zip_name = "public.zip" 

    # 1. Download the zip file (this caches the zip locally in ~/.cache/huggingface)
    zip_path = hf_hub_download(repo_id=repo_id, repo_type="dataset", filename=zip_name, token=token)
    
    # 2. Open the ZIP and load the specific image into memory
    with zipfile.ZipFile(zip_path, 'r') as z:
        # Some zips have paths inside, so you may need to search the zip info for the filename
        matching_files = [f for f in z.namelist() if f.endswith(filename)]
        if not matching_files:
            raise FileNotFoundError(f"File {filename} not found inside {zip_name}")
            
        with z.open(matching_files[0]) as f:
            return Image.open(io.BytesIO(f.read())).convert("RGB")

In [13]:
target_filenames = [
    "pubmed/00_39_PMC10381143_jimaging-09-00148-g011_0.jpg",
    "pubmed/00_39_PMC10381143_jimaging-09-00148-g011_1.jpg",
    "pubmed/00_39_PMC10381143_jimaging-09-00148-g012_1.jpg"
]

# Fetch all filenames from the training split (this might take a few seconds)
all_filenames = ds["train"]["filename"]

data = []

print("Searching for images in the dataset...\n")

for target in target_filenames:
    found = False
    
    # Iterate through all filenames to find a substring match
    for idx, ds_filename in enumerate(all_filenames):
        if ds_filename and target in ds_filename:
            
            # Format it exactly as requested in your data list
            # Prepending 'derm_train_images/' to match local paths
            data.append({
                "filename": f"derm_train_images/{target}", 
                "index": idx
            })
            
            found = True
            break  # Move to the next target once found
            
    if not found:
        print(f"[!] Not found: {target}")

# Print the cleanly formatted list
import json
print("data =", json.dumps(data, indent=4))

Searching for images in the dataset...

data = [
    {
        "filename": "derm_train_images/pubmed/00_39_PMC10381143_jimaging-09-00148-g011_0.jpg",
        "index": 202352
    },
    {
        "filename": "derm_train_images/pubmed/00_39_PMC10381143_jimaging-09-00148-g011_1.jpg",
        "index": 151106
    },
    {
        "filename": "derm_train_images/pubmed/00_39_PMC10381143_jimaging-09-00148-g012_1.jpg",
        "index": 8350
    }
]


In [11]:
print(ds["train"][data[0]["index"]])

{'filename': 'pubmed/00_39_PMC10381143_jimaging-09-00148-g011_0.jpg', 'caption': '(a) Dermoscopic image.', 'truncated_caption': '(a) Dermoscopic image.', 'source': 'pubmed_fail', 'source_type': 'knowledge', 'disease_label': 'no definitive diagnosis', 'hierarchical_disease_label': None, 'skin_concept': 'No visual concepts', 'body_location': 'No body location information', 'symptoms': 'No symptom information', 'age': 'No age information', 'gender': 'No gender information'}


In [12]:
# Define your list of models to test
# Format: List of dictionaries with model_name and its library backend
models_to_test = [
    {"name": "hf-hub:redlessone/DermLIP_ViT-B-16", "backend": "open_clip"},
    {"name": "openai/clip-vit-base-patch32", "backend": "huggingface"}
]

# Get a few sample indices to run
results = {}

for model_info in models_to_test:
    model_name = model_info["name"]
    backend = model_info["backend"]
    print(f"\n[{model_name}] Loading Model...")

    # ==========================================
    # 1. Load Model dynamically based on backend
    # ==========================================
    if backend == "huggingface":
        model = AutoModel.from_pretrained(model_name).to('cuda').eval()
        processor = AutoProcessor.from_pretrained(model_name)
        tokenizer = processor # HF uses combined processors
    else: # open_clip
        model, _, processor = open_clip.create_model_and_transforms(model_name)
        model = model.to('cuda').eval()
        tokenizer = open_clip.get_tokenizer(model_name)
        
        # OpenCLIP needs manual patch_size extraction
        patch_size = 16 
        if hasattr(model.visual, 'patch_size'):
            if isinstance(model.visual.patch_size, tuple):
                patch_size = model.visual.patch_size[0]
            else:
                patch_size = model.visual.patch_size

    # Initialize results dict for this model
    results[model_name] = []

    # ==========================================
    # 2. Iterate over samples
    # ==========================================
    for item in data:
        img_path = item["filename"]
        idx = item["index"]
        print(f"  -> Processing sample {idx}")
        sample = ds["train"][idx]
        input_text = sample["caption"]
                
        # If the image hasn't been downloaded physically yet, you might need to handle it.
        # Assuming they are physically located at `derm_train_images/`:
        try:
            input_image = Image.open(img_path).convert("RGB")
        except FileNotFoundError:
            print(f"     [!] Image not found at {img_path}. Skipping.")
            continue

        # ==========================================
        # 3. Create the appropriate Game
        # ==========================================
        if backend == "huggingface":
            game = src.game_huggingface.VisionLanguageGame(
                model=model,
                processor=processor,
                input_image=input_image,
                input_text=input_text,
                batch_size=32 
            )
        else: # open_clip
            game = src.game_openclip.OpenCLIPGame(
                model=model,
                image_processor=processor,
                text_tokenizer=tokenizer,
                input_image=input_image,
                input_text=input_text,
                patch_size=patch_size,
                batch_size=32 
            )

        print(f"     [ Game Initialized ] Modalities: TextPlayers={game.n_players_text}, ImagePlayers={game.n_players_image}")

        # ==========================================
        # 4. Compute Explanations via FIxLIP framework
        # ==========================================
        approximator = src.fixlip.FIxLIP(
            n_players_text=game.n_players_text,
            n_players_image=game.n_players_image,
            random_state=42
        )
        
        # Compute the cross-modal interaction values (ProxySHAP for speed)
        interaction_values = approximator.approximate_crossmodal(
            game=game,
            budget_image=2**11, # 2048 samples
            budget_text=2**6,   # 64 samples
            approximation_type="proxyshap"
        )
        
        # Save results mapping
        results[model_name].append({
            "idx": idx,
            "filename": sample["filename"],
            "interaction_values": interaction_values
        })
        
    print(f"[{model_name}] Finished!\n")
    
    del model
    del processor
    if backend == "open_clip":
        del tokenizer
    torch.cuda.empty_cache()


[hf-hub:redlessone/DermLIP_ViT-B-16] Loading Model...
  -> Processing sample 202352
PAD_TOKEN:  0
     [ Game Initialized ] Modalities: TextPlayers=8, ImagePlayers=196
  -> Processing sample 151106
PAD_TOKEN:  0
     [ Game Initialized ] Modalities: TextPlayers=19, ImagePlayers=196


KeyboardInterrupt: 

In [13]:
print(results)

{'hf-hub:redlessone/DermLIP_ViT-B-16': [{'idx': 202352, 'filename': 'pubmed/00_39_PMC10381143_jimaging-09-00148-g011_0.jpg', 'interaction_values': InteractionValues(
    index=FBII, max_order=2, min_order=0, estimated=True, estimation_budget=131072,
    n_players=204, baseline_value=24.718517303466797
)}, {'idx': 151106, 'filename': 'pubmed/00_39_PMC10381143_jimaging-09-00148-g011_1.jpg', 'interaction_values': InteractionValues(
    index=FBII, max_order=2, min_order=0, estimated=True, estimation_budget=131072,
    n_players=215, baseline_value=24.280134201049805
)}, {'idx': 8350, 'filename': 'pubmed/00_39_PMC10381143_jimaging-09-00148-g012_1.jpg', 'interaction_values': InteractionValues(
    index=FBII, max_order=2, min_order=0, estimated=True, estimation_budget=131072,
    n_players=208, baseline_value=24.3498477935791
)}], 'openai/clip-vit-base-patch32': [{'idx': 202352, 'filename': 'pubmed/00_39_PMC10381143_jimaging-09-00148-g011_0.jpg', 'interaction_values': InteractionValues(
   